# Time-resolved multivariate RPE category (3-class)

Crossnobis distance at feedback with **Freedman–Lane** permutation nulls on one-hot labels residualized on covariates (`zpm`). **Pairwise** and **canonical** LDA filters; cue-time projection with surrogate z-scoring. Per-electrode **Haufe** trial contributions saved alongside encoding.

Requires: `mne`, `sklearn`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `joblib`, and project `mv_regression_utils` / data paths.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

from plot_config import set_plot_style
from rpe_cat_mvpa_utils import (
    EncodingConfig,
    ProjectionConfig,
    process_subject_encoding,
    process_subject_projection,
    run_encoding_pipeline,
    run_projection_pipeline,
    save_projection_npz,
    load_electrode_contribs,
    summarize_electrode_contribs,
    compute_omnibus_reactivation,
)

set_plot_style()


In [ ]:
# =============================
# Paths + analysis configuration
# =============================
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Subject list (edit or load from database like template notebook)
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
try:
    demo_data = pd.read_excel(demo_file, engine='openpyxl')
    subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)]
    subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
    subjects = subj_df.MSSMCode.astype(str).tolist()
except Exception:
    subjects = ['MS036']  # fallback if xlsx unavailable

ROIS = ['OFC', 'ACC', 'HPC', 'dmPFC', 'dlPFC', 'AMY']
PREDICTOR = 'rpe_cat'
COVARIATES = ()
EVENT_ENCODING = 'feedback_start'
EVENT_PROJECTION = 'cue_on'
REGRESSOR_SOURCE = 'learn'

FREQ_RANGE = (70., 201.)
WIN_MS = 200
DOWNSAMPLE_FACTOR = 5
MIN_TRIALS = 10
SEED = 42
N_PERM = 250
N_SURROGATES = 250
N_JOBS_OUTER = min(8, os.cpu_count() or 1)
BAND = 'hfa'

OUT_DIR = os.path.join(load_dir, 'rpe_cat_mvpa_outputs')
ENC_CSV = os.path.join(OUT_DIR, f'mv_rpe_cat_encoding_{EVENT_ENCODING}_{BAND}.csv')
CONTRIB_DIR = os.path.join(OUT_DIR, 'electrode_contribs')
PROJ_DIR = os.path.join(OUT_DIR, 'projection_npz')
CHK_ENC = os.path.join(OUT_DIR, 'checkpoints_encoding')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CONTRIB_DIR, exist_ok=True)
os.makedirs(PROJ_DIR, exist_ok=True)

enc_cfg = EncodingConfig(
    event=EVENT_ENCODING,
    predictor=PREDICTOR,
    covariates=COVARIATES,
    freq_range=FREQ_RANGE,
    win_ms=WIN_MS,
    downsample_factor=DOWNSAMPLE_FACTOR,
    n_folds=10,
    n_perm=N_PERM,
    min_trials=MIN_TRIALS,
    seed=SEED,
    regressor_source=REGRESSOR_SOURCE,
)
proj_cfg = ProjectionConfig(
    event_encoding=EVENT_ENCODING,
    event_projection=EVENT_PROJECTION,
    predictor=PREDICTOR,
    covariates=COVARIATES,
    freq_range=FREQ_RANGE,
    win_ms=WIN_MS,
    downsample_factor=DOWNSAMPLE_FACTOR,
    min_trials=MIN_TRIALS,
    n_surrogates=N_SURROGATES,
    seed=SEED,
    regressor_source=REGRESSOR_SOURCE,
)

SMOKE_SUBJECT = 'MS036' if 'MS036' in subjects else subjects[0]
SMOKE_ROI = 'ACC' if 'ACC' in ROIS else ROIS[0]
print('Subjects:', len(subjects), '| Smoke:', SMOKE_SUBJECT, SMOKE_ROI)


In [ ]:
# =============================
# Smoke: encoding + sidecar NPZ
# =============================
df_smoke, sc = process_subject_encoding(
    SMOKE_SUBJECT, base_dir, load_dir, SMOKE_ROI, enc_cfg, verbose=True,
)
print(df_smoke.head())
print('sidecar' if sc is not None else 'no sidecar', '| rows', len(df_smoke))
if sc is not None:
    print('contribs_pw', sc['contribs_pairwise_haufe'].shape)
    import json as _json
    _p = os.path.join(CONTRIB_DIR, f'{SMOKE_SUBJECT}_{SMOKE_ROI}_rpe_cat_electrode_contribs.npz')
    np.savez_compressed(
        _p,
        contribs_pairwise_haufe=sc['contribs_pairwise_haufe'],
        contribs_canonical_haufe=sc['contribs_canonical_haufe'],
        pair_labels=sc['pair_labels'],
        feedback_times=sc['feedback_times'],
        electrode_labels=sc['electrode_labels'],
        class_labels=sc['class_labels'],
        trial_index=sc['trial_index'],
        meta_json=_json.dumps(sc['meta']),
    )
    print('Wrote', _p)
if len(df_smoke):
    df_smoke.to_csv(os.path.join(OUT_DIR, f'_smoke_encoding_{SMOKE_SUBJECT}_{SMOKE_ROI}.csv'), index=False)


In [ ]:
# =============================
# Smoke: projection NPZ
# =============================
out_proj = process_subject_projection(
    SMOKE_SUBJECT, base_dir, load_dir, SMOKE_ROI, proj_cfg, verbose=True,
)
if out_proj:
    print({k: (v.shape if hasattr(v, 'shape') else type(v)) for k, v in out_proj.items() if isinstance(v, np.ndarray)})
    _pp = os.path.join(PROJ_DIR, SMOKE_ROI, f'{SMOKE_SUBJECT}_rpe_cat_projection.npz')
    save_projection_npz(_pp, out_proj, proj_cfg)
    print('Wrote', _pp)
else:
    print('Projection skipped (no data)')


In [ ]:
# =============================
# Batch encoding (all subjects × ROIs)
# =============================
RUN_BATCH_ENCODING = True  # set True for full run

if RUN_BATCH_ENCODING:
    enc_df = run_encoding_pipeline(
        subjects=subjects,
        rois=ROIS,
        base_dir=base_dir,
        load_dir=load_dir,
        cfg=enc_cfg,
        out_csv=ENC_CSV,
        contribs_dir=CONTRIB_DIR,
        n_jobs=N_JOBS_OUTER,
        checkpoint_dir=CHK_ENC,
        overwrite_checkpoints=True,
        verbose=True,
    )
    print('Saved', ENC_CSV, 'rows', len(enc_df))
else:
    print('Set RUN_BATCH_ENCODING=True to run batch encoding.')


In [ ]:
# =============================
# Batch projection
# =============================
RUN_BATCH_PROJECTION = True

if RUN_BATCH_PROJECTION:
    run_projection_pipeline(
        subjects=subjects,
        rois=ROIS,
        base_dir=base_dir,
        load_dir=load_dir,
        cfg=proj_cfg,
        out_dir=PROJ_DIR,
        n_jobs=N_JOBS_OUTER,
        verbose=True,
    )
    print('Projection NPZs under', PROJ_DIR)
else:
    print('Set RUN_BATCH_PROJECTION=True to run batch projection.')


In [ ]:
# =============================
# Group encoding: crossnobis z vs time (6 ROIs)
# =============================
from mne.stats import permutation_cluster_1samp_test

def plot_group_encoding(model_df, plot_col='z_metric', n_perm_cluster=5000):
    roi_order = ['OFC', 'ACC', 'HPC', 'dmPFC', 'dlPFC', 'AMY']
    roi_pos = {'OFC': (0,0), 'ACC': (0,1), 'HPC': (0,2), 'dmPFC': (1,0), 'dlPFC': (1,1), 'AMY': (1,2)}
    roi_colors = {
        'HPC': np.array([0.44, 0.29, 1.0]), 'AMY': np.array([0.49, 0, 0.9]),
        'OFC': np.array([1, 0.76, 0.42]), 'ACC': np.array([0.71, 0.27, 0.68]),
        'dmPFC': np.array([0.41, 0.02, 0]), 'dlPFC': np.array([1, 0.06, 0.06]),
    }
    fig, ax = plt.subplots(2, 3, figsize=(20, 10), sharey=True)
    for region in roi_order:
        ix, jx = roi_pos[region]
        color = roi_colors.get(region, [0.3]*3)
        rd = model_df[model_df['region'] == region].copy()
        if rd.empty:
            ax[ix,jx].set_title(f'{region} (no data)'); ax[ix,jx].axis('off'); continue
        tsu = np.sort(rd['ts'].unique())
        m = rd.groupby('ts')[plot_col].mean().reindex(tsu).to_numpy()
        se = rd.groupby('ts')[plot_col].sem().reindex(tsu).to_numpy()
        ax[ix,jx].plot(tsu, m, color=color, lw=1.5)
        ax[ix,jx].fill_between(tsu, m-np.nan_to_num(se), m+np.nan_to_num(se), color=color, alpha=0.2)
        subj_time = rd.pivot_table(index='subj', columns='ts', values=plot_col, aggfunc='mean').reindex(columns=tsu).dropna(how='any', axis=0)
        if subj_time.shape[0] >= 2:
            try:
                T_obs, clusters, pvals, _ = permutation_cluster_1samp_test(
                    subj_time.to_numpy(dtype=np.float64), n_permutations=n_perm_cluster, tail=1, verbose=False)
                for ic, c in enumerate(clusters):
                    if pvals[ic] <= 0.05:
                        c_ix = c[0]; c_ts = tsu[c_ix]
                        if len(c_ts):
                            y0, y1 = ax[ix,jx].get_ylim()
                            ax[ix,jx].fill_between(c_ts, y0+0.88*(y1-y0), y0+0.96*(y1-y0), color=color, alpha=0.35)
            except Exception as e:
                print(region, e)
        ax[ix,jx].axhline(0, color='k', ls='--', lw=1)
        ax[ix,jx].set(title=region, xlabel='time (ms)' if ix==1 else '', ylabel=plot_col if jx==0 else '')
        sns.despine(ax=ax[ix,jx])
    plt.suptitle(f'Group MV RPE-cat encoding | crossnobis z (FL perm null) | n={model_df.subj.nunique()}')
    plt.tight_layout()
    plt.show()

_csv_try = [ENC_CSV, os.path.join(OUT_DIR, f'_smoke_encoding_{SMOKE_SUBJECT}_{SMOKE_ROI}.csv')]
_csv = next((c for c in _csv_try if os.path.exists(c)), None)
if _csv is not None:
    _mdf = pd.read_csv(_csv)
    _pc = 'z_metric' if 'z_metric' in _mdf.columns else 'metric'
    plot_group_encoding(_mdf, plot_col=_pc)
else:
    print('Run smoke encoding first; no CSV at', _csv_try)


In [ ]:
# Group encoding plot is defined and run in the cell above.
pass


In [ ]:
# =============================
# Omnibus reactivation heatmap (single map per ROI)
# Sum of squared z-reactivation across canonical LDA axes (~ Chi^2(2) under null).
# =============================

def load_proj_npz(roi, subj):
    p = os.path.join(PROJ_DIR, roi, f'{subj}_rpe_cat_projection.npz')
    if not os.path.exists(p):
        return None
    return np.load(p, allow_pickle=True)

def plot_omnibus_reactivation_heatmap(roi, subjects_used, center_chi2=True):
    """One (feedback x cue) heatmap: group mean omnibus reactivation."""
    maps = []
    for s in subjects_used:
        z = load_proj_npz(roi, s)
        if z is None or 'z_reactivation_canonical' not in z.files:
            continue
        zc = np.asarray(z['z_reactivation_canonical'], dtype=np.float64)
        om = compute_omnibus_reactivation(zc)
        maps.append(np.nanmean(om, axis=2))
    if not maps:
        print('No NPZ for', roi)
        return
    D = np.stack(maps, axis=0)
    mean_m = np.nanmean(D, axis=0)
    ref = np.load(os.path.join(PROJ_DIR, roi, f'{subjects_used[0]}_rpe_cat_projection.npz'), allow_pickle=True)
    fb_t = ref['feedback_times']
    cue_t = ref['cue_times']
    plot_data = mean_m - 2.0 if center_chi2 else mean_m
    vmax = float(np.nanmax(np.abs(plot_data))) if center_chi2 else float(np.nanmax(plot_data))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    fig, ax = plt.subplots(figsize=(5, 4))
    cmap = 'RdBu_r' if center_chi2 else 'magma'
    vmin = -vmax if center_chi2 else 0.0
    im = ax.imshow(
        plot_data, origin='lower', aspect='auto',
        extent=[float(cue_t.min()), float(cue_t.max()), float(fb_t.min()), float(fb_t.max())],
        cmap=cmap, vmin=vmin, vmax=vmax,
    )
    ax.set_xlabel('cue time (s)')
    ax.set_ylabel('feedback time (s)')
    ttl = f'{roi} | omnibus reactivation (sum of z^2 on canonical axes)'
    if center_chi2:
        ttl += r' | map − 2 ($\approx$ E[Chi$^2$(2)] under null)'
    ax.set_title(ttl)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

# Optional: per-pairwise-filter maps for interpretation (set True to plot)
PLOT_PAIRWISE_APPENDIX = False

def plot_pairwise_appendix(roi, subjects_used):
    maps = []
    for s in subjects_used:
        z = load_proj_npz(roi, s)
        if z is None or 'z_reactivation_pairwise' not in z.files:
            continue
        arr = np.nanmean(z['z_reactivation_pairwise'], axis=2)
        maps.append(arr)
    if not maps:
        return
    mean_m = np.nanmean(np.stack(maps, axis=0), axis=0)
    ref = np.load(os.path.join(PROJ_DIR, roi, f'{subjects_used[0]}_rpe_cat_projection.npz'), allow_pickle=True)
    fb_t, cue_t = ref['feedback_times'], ref['cue_times']
    n_filt = mean_m.shape[-1]
    fig, axes = plt.subplots(1, n_filt, figsize=(4 * n_filt, 4), squeeze=False)
    for fi in range(n_filt):
        ax = axes[0, fi]
        v = float(np.nanmax(np.abs(mean_m[..., fi]))) or 1.0
        im = ax.imshow(
            mean_m[..., fi], origin='lower', aspect='auto',
            extent=[float(cue_t.min()), float(cue_t.max()), float(fb_t.min()), float(fb_t.max())],
            cmap='RdBu_r', vmin=-v, vmax=v,
        )
        ax.set_xlabel('cue time (s)')
        ax.set_ylabel('feedback time (s)')
        ax.set_title(f'{roi} pairwise filter {fi}')
        plt.colorbar(im, ax=ax, shrink=0.7)
    plt.suptitle('Appendix: pairwise z-reactivation (optional)')
    plt.tight_layout()
    plt.show()

_subj_list = [SMOKE_SUBJECT] if os.path.exists(os.path.join(PROJ_DIR, SMOKE_ROI, f'{SMOKE_SUBJECT}_rpe_cat_projection.npz')) else []
if _subj_list:
    plot_omnibus_reactivation_heatmap(SMOKE_ROI, _subj_list, center_chi2=True)
    if PLOT_PAIRWISE_APPENDIX:
        plot_pairwise_appendix(SMOKE_ROI, _subj_list)
else:
    print('Run smoke projection or set PROJ_DIR paths after batch.')



In [ ]:
# Haufe pattern bar chart (time-averaged |weights| from projection NPZ — use patterns from saved file)
def plot_haufe_bars_smoke():
    z = load_proj_npz(SMOKE_ROI, SMOKE_SUBJECT)
    if z is None:
        print('no npz'); return
    P = z['patterns_pairwise']  # (n_fb, n_elec, 3)
    labs = z['electrode_labels']
    mean_abs = np.nanmean(np.abs(P), axis=0)  # (n_elec, 3)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
    for k in range(3):
        ax[k].bar(np.arange(len(labs)), mean_abs[:, k])
        ax[k].set_xticks(np.arange(len(labs)))
        ax[k].set_xticklabels(labs, rotation=90, fontsize=7)
        ax[k].set_title(f'pair {k}')
    plt.suptitle(f'Haufe |time-mean pattern| — {SMOKE_SUBJECT} {SMOKE_ROI}')
    plt.tight_layout()
    plt.show()

plot_haufe_bars_smoke()


In [ ]:
# =============================
# Memory: Hits − Misses on omnibus reactivation (canonical z^2 sum per trial)
# =============================

def load_proj_npz(roi, subj):
    p = os.path.join(PROJ_DIR, roi, f'{subj}_rpe_cat_projection.npz')
    if not os.path.exists(p):
        return None
    return np.load(p, allow_pickle=True)

def hits_minus_misses_group(roi, subjects_used):
    diffs = []
    for s in subjects_used:
        z = load_proj_npz(roi, s)
        if z is None or 'memory_outcomes' not in z.files:
            continue
        if 'z_reactivation_canonical' not in z.files:
            continue
        mem = z['memory_outcomes'].astype(str)
        zc = np.asarray(z['z_reactivation_canonical'], dtype=np.float64)
        om = compute_omnibus_reactivation(zc)
        hits = mem == 'hits'
        miss = mem == 'misses'
        if hits.sum() == 0 or miss.sum() == 0:
            continue
        d = np.nanmean(om[:, :, hits], axis=2) - np.nanmean(om[:, :, miss], axis=2)
        diffs.append(d)
    if not diffs:
        print('no data')
        return
    D = np.stack(diffs, axis=0)
    m = np.nanmean(D, axis=0)
    ref = np.load(os.path.join(PROJ_DIR, roi, f'{subjects_used[0]}_rpe_cat_projection.npz'))
    fb_t, cue_t = ref['feedback_times'], ref['cue_times']
    v = float(np.nanmax(np.abs(m))) or 1.0
    plt.figure(figsize=(5, 4))
    plt.imshow(m, origin='lower', aspect='auto',
        extent=[float(cue_t.min()), float(cue_t.max()), float(fb_t.min()), float(fb_t.max())],
        cmap='RdBu_r', vmin=-v, vmax=v)
    plt.colorbar()
    plt.title(f'Group mean Hits−Misses (omnibus) | {roi}')
    plt.xlabel('cue (s)')
    plt.ylabel('feedback (s)')
    plt.tight_layout()
    plt.show()

if _subj_list:
    hits_minus_misses_group(SMOKE_ROI, _subj_list)



In [ ]:
# =============================
# Export trial × electrode × pair Haufe contributions (merge-ready)
# =============================
EXPORT_WINDOW_MS = (300.0, 800.0)  # feedback time ms

def export_contribs_long(subj, roi, t_lo, t_hi):
    path = os.path.join(CONTRIB_DIR, f'{subj}_{roi}_rpe_cat_electrode_contribs.npz')
    if not os.path.exists(path):
        print('missing', path); return None
    L = load_electrode_contribs(subj, roi, CONTRIB_DIR)
    return summarize_electrode_contribs(subj, roi, L, t_lo, t_hi, agg='mean')

df_long = export_contribs_long(SMOKE_SUBJECT, SMOKE_ROI, *EXPORT_WINDOW_MS)
if df_long is not None and len(df_long):
    out_csv = os.path.join(OUT_DIR, f'haufe_long_{SMOKE_SUBJECT}_{SMOKE_ROI}.csv')
    df_long.to_csv(out_csv, index=False)
    print('Wrote', out_csv, 'rows', len(df_long))
    print(df_long.head())
else:
    print('Run encoding smoke first.')
